# Semantic Chunking

## Overview

Standard chunking splits text at fixed character counts — it doesn't care if it cuts a sentence in half. **Semantic chunking** splits text at natural breakpoints where the *meaning* changes.

| Fixed-Size Chunking | Semantic Chunking |
|---|---|
| Splits every N characters | Splits where **meaning shifts** |
| May cut mid-sentence or mid-idea | Keeps complete ideas together |
| Chunks may lack coherence | Chunks are semantically coherent |

First proposed by [Greg Kamradt](https://youtu.be/8OJC21T2SL4?t=1933), implemented in [LangChain](https://python.langchain.com/docs/how_to/semantic-chunker/).

## How It Works

1. Embed every sentence in the document
2. Measure the cosine similarity between consecutive sentence embeddings
3. When the similarity **drops sharply**, that's a semantic boundary — split there

## Breakpoint Types

| Type | Split when... |
|---|---|
| `percentile` | Difference between sentences > X-th percentile of all differences |
| `standard_deviation` | Difference > X standard deviations above the mean |
| `interquartile` | Uses interquartile range to find outlier differences |

## Models Used

- **Embeddings**: `mxbai-embed-large:335m` via Ollama (used for both chunking and retrieval)

<div style="text-align: center;">

<img src="./images/semantic_chunking_comparison.svg" alt="Semantic Chunking" style="width:100%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
import fitz
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings
from IPython.display import display, HTML

---
## Step 1: Set Up the Embedding Model

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("Embedding model ready")

Embedding model ready


---
## Step 2: Load the PDF as a Single String

Unlike other notebooks where we use `PyPDFLoader` (which gives one Document per page), here we need the **entire document as one string** so the semantic chunker can analyze sentence-to-sentence transitions across the whole text.

In [3]:
path = "data/Understanding_Climate_Change.pdf"

doc = fitz.open(path)
content = ""
for page_num in range(len(doc)):
    page = doc[page_num]
    content += page.get_text()

print(f"Loaded {len(doc)} pages")
print(f"Total text length: {len(content)} characters")
print(f"\nFirst 300 characters:\n{content[:300]}...")

Loaded 33 pages
Total text length: 72561 characters

First 300 characters:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an exte...


---
## Step 3: Apply Semantic Chunking

The `SemanticChunker` from LangChain:
1. Embeds each sentence
2. Computes similarity between consecutive sentences
3. Splits where the similarity drops below the threshold

We use the **percentile** method with a threshold of 70 — meaning splits happen where the sentence-to-sentence difference is in the top 30% of all differences.

In [4]:
text_splitter = SemanticChunker(
    embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)

chunks = text_splitter.create_documents([content])

print(f"Created {len(chunks)} semantic chunks")
print(f"\nChunk sizes (characters):")
for i, chunk in enumerate(chunks[:5]):
    print(f"  Chunk {i}: {len(chunk.page_content)} chars")
if len(chunks) > 5:
    print(f"  ... ({len(chunks) - 5} more)")

Created 185 semantic chunks

Chunk sizes (characters):
  Chunk 0: 465 chars
  Chunk 1: 745 chars
  Chunk 2: 95 chars
  Chunk 3: 704 chars
  Chunk 4: 94 chars
  ... (180 more)


Notice how chunk sizes **vary** — unlike fixed-size chunking where every chunk is the same length. Semantic chunking creates chunks based on meaning, so some are short (a single idea) and others are longer (a complex topic).

---
## Step 4: Build the Vector Store and Retriever

In [5]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Vector store created with {len(chunks)} chunks")
print(f"Retriever set to return top 2 results")

Vector store created with 185 chunks
Retriever set to return top 2 results


---
## Step 5: Test the Retriever

In [6]:
test_query = "What is the main cause of climate change?"
print(f"Query: {test_query}\n")

results = retriever.invoke(test_query)

for i, doc in enumerate(results):
    display(HTML(f"<span style='background-color: lightblue;'> <b>Context {i}:</b></span>"))
    display(HTML(f"<b>Page Content:</b> {doc.page_content}"))
    print("=" * 80)

Query: What is the main cause of climate change?



---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up embedding model |
| 2 | Loaded entire PDF as one string |
| 3 | **Semantic chunking** — split at meaning boundaries (percentile method, threshold=70) |
| 4 | Built FAISS vector store from semantic chunks |
| 5 | Tested retrieval |

**Key insight:** Semantic chunks have **variable sizes** because they follow the natural structure of the text. A short paragraph about one topic becomes one chunk; a long explanation of a complex concept stays together instead of being split arbitrarily. This leads to more coherent retrieval results, especially for documents with varied information density.